# ValetAI Data Pipeline
## Dataset Generation + Bronze/Silver/Gold

In [0]:
# Creating waste collection data

import random
from datetime import datetime, timedelta
import pandas as pd

random.seed(42)

properties = ["Sunset Ridge", "Maple Court", "Green Oaks",
              "Birchwood Commons", "Lakeview Terrace",
              "Cedar Park", "Willow Creek", "Harbor Pointe"]
regions = ["Southeast", "Midwest", "Northeast", "West"]
routes  = [f"R{i}" for i in range(10, 25)]

rows = []
base = datetime(2025, 1, 1)

for i in range(8000):
    pdate  = base + timedelta(days=random.randint(0, 180))
    missed = random.choices([0,1,2,3], weights=[70,15,10,5])[0]
    prop   = random.choice(properties)
    rows.append({
        "collection_id":   f"C{i+1:06d}",
        "property_id":     f"P{properties.index(prop)+1:03d}",
        "property_name":   prop,
        "collection_date": pdate.strftime("%Y-%m-%d"),
        "route_id":        random.choice(routes),
        "bags_collected":  random.randint(40, 160),
        "missed_pickups":  missed,
        "completion_time": f"{random.randint(14,22)}:{random.randint(0,59):02d}",
        "collector_id":    f"D{random.randint(1,20):02d}",
        "region":          random.choice(regions),
    })

pdf_logs = pd.DataFrame(rows)
df_logs  = spark.createDataFrame(pdf_logs)

print(f"Created {df_logs.count()} rows")
display(df_logs.limit(5))

Created 8000 rows


collection_id,property_id,property_name,collection_date,route_id,bags_collected,missed_pickups,completion_time,collector_id,region
C000001,P005,Lakeview Terrace,2025-06-13,R13,68,0,16:47,D04,Southeast
C000002,P001,Sunset Ridge,2025-06-01,R11,67,0,17:32,D20,Southeast
C000003,P007,Willow Creek,2025-05-24,R13,97,0,18:51,D01,Midwest
C000004,P005,Lakeview Terrace,2025-06-28,R12,67,0,19:06,D03,West
C000005,P006,Cedar Park,2025-01-25,R19,73,0,14:46,D15,Southeast


In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta

# Issue Templates

issue_templates = {
    "Gate Access": [
        "Driver could not access community - gate code expired",
        "Gate access restrictions caused delay on route",
        "Leasing office did not answer for gate code reset",
        "Security denied entry due to outdated access credentials",
        "Community gate remained locked during scheduled pickup window",
        "Temporary gate malfunction prevented collector entry",
        "Access card failed to unlock service entrance",
        "Property manager unavailable to provide updated gate code",
        "New security protocol delayed collector access",
        "Multiple access attempts required before entry was granted",
        "Incorrect gate code provided by property management",
        "Gate intercom system malfunctioned during service hours",
        "Visitor access restrictions delayed collection process",
        "Access credentials expired unexpectedly",
        "Manual gate operation caused route delays",
        "Restricted access due to special community event",
        "Delayed authorization from security personnel",
        "Entry denied due to missing vendor registration",
        "Service entrance blocked by maintenance activity",
        "Community access policy changes were not communicated"
    ],

    "Staffing": [
        "Staff shortage on route caused missed pickups",
        "Collector called in sick, route covered late",
        "New collector unfamiliar with property layout",
        "Insufficient staffing levels increased route completion time",
        "Unexpected employee absence impacted daily operations",
        "Training requirements delayed assignment of replacement collector",
        "Route coverage reduced due to labor constraints",
        "Peak demand exceeded available workforce capacity",
        "Shift scheduling conflict resulted in delayed collections",
        "High employee turnover affected route consistency",
        "Insufficient weekend staffing caused backlog",
        "Overtime limitations reduced available coverage",
        "Staff reassignment impacted scheduled routes",
        "Temporary workers required additional supervision",
        "Multiple employee absences occurred simultaneously",
        "Resource allocation issue affected collection schedule",
        "Reduced workforce availability impacted service quality",
        "Holiday staffing shortages caused delays",
        "Unexpected resignations affected operations",
        "Recruitment delays limited route coverage"
    ],

    "Vehicle": [
        "Vehicle breakdown impacted collections on route",
        "Truck mechanical issue delayed entire route schedule",
        "Collection vehicle required mid-route maintenance stop",
        "Engine overheating forced temporary route suspension",
        "Flat tire caused service delays across multiple properties",
        "Vehicle inspection identified safety concerns before dispatch",
        "Fuel system malfunction impacted collection operations",
        "Replacement vehicle arrived later than expected",
        "Unexpected maintenance increased route completion time",
        "Fleet availability issues caused route reassignment",
        "Battery failure delayed route departure",
        "Hydraulic lift malfunction slowed operations",
        "Vehicle servicing overlapped with scheduled collections",
        "Brake system inspection delayed deployment",
        "Transmission issue impacted route completion",
        "Equipment failure reduced collection efficiency",
        "Fleet shortage increased route workloads",
        "Emergency repair interrupted collection schedule",
        "Vehicle diagnostics required immediate attention",
        "Maintenance backlog reduced fleet readiness"
    ],

    "Resident Behavior": [
        "Resident complaints increased after route schedule change",
        "Bins not placed at doorstep for scheduled pickup",
        "Resident left bin blocked by parked vehicle",
        "Improper waste disposal created collection challenges",
        "Residents placed bins outside designated collection hours",
        "Overflowing bins required additional collection effort",
        "Several residents reported confusion regarding pickup schedule",
        "Unauthorized items were left for collection",
        "Missed communication led to resident dissatisfaction",
        "Residents failed to follow waste segregation guidelines",
        "Resident participation declined during holiday period",
        "Collection instructions were frequently ignored",
        "Improperly secured waste created operational issues",
        "Residents requested unscheduled pickups",
        "Excess waste volume exceeded expected capacity",
        "Resident complaints were concentrated in specific buildings",
        "Repeated policy violations impacted service quality",
        "Collection bins were inaccessible during service window",
        "Late bin placement caused missed pickups",
        "Resident education on waste procedures was recommended"
    ],

    "Weather": [
        "Heavy rain delayed collection schedule across region",
        "Snow accumulation blocked access to several units",
        "Severe weather reduced operational efficiency",
        "Flooded access roads prevented route completion",
        "High winds created safety concerns for collectors",
        "Storm conditions resulted in temporary service suspension",
        "Extreme temperatures slowed collection activities",
        "Weather advisory required route modification",
        "Lightning risk delayed evening collections",
        "Adverse weather increased average completion times",
        "Ice accumulation restricted vehicle movement",
        "Poor visibility affected route safety",
        "Weather disruptions impacted multiple communities",
        "Unexpected storm activity required schedule changes",
        "Wet conditions reduced collection speed",
        "Severe heat affected workforce productivity",
        "Weather-related road closures impacted routes",
        "Emergency weather response delayed operations",
        "Snow removal activities blocked service access",
        "Regional weather event affected service levels"
    ],

    "Route Planning": [
        "Route optimization issue increased travel time",
        "Traffic congestion delayed collections across multiple properties",
        "Inefficient route sequencing reduced productivity",
        "Construction activity blocked primary collection route",
        "GPS routing error caused missed service locations",
        "Unexpected road closure required route diversion",
        "Route balancing adjustments improved service coverage",
        "High-density properties created route bottlenecks",
        "Scheduling overlap reduced operational efficiency",
        "Route redesign recommended to improve performance",
        "Peak-hour traffic increased collection duration",
        "Inefficient property sequencing impacted schedule adherence",
        "Navigation issue delayed route completion",
        "Route capacity exceeded planned workload",
        "Property clustering analysis suggested optimization opportunities",
        "Travel distance increased due to route modifications",
        "Service area expansion affected route timing",
        "Route handoff between teams caused delays",
        "Operational review identified route inefficiencies",
        "Dynamic route adjustments improved service outcomes"
    ],

    "Property Management": [
        "Property management requested schedule adjustment",
        "Management communication delay affected operations",
        "Maintenance work restricted access to collection areas",
        "Property renovation disrupted normal collection process",
        "Manager reported recurring service concerns",
        "Operational changes were not communicated in advance",
        "Building access procedures updated without notice",
        "Property staffing changes impacted coordination",
        "Management requested additional pickup support",
        "Service expectations reviewed with property leadership",
        "Property inspection activities disrupted collections",
        "Management escalation required operational review",
        "Community event impacted collection schedule",
        "Property policy changes affected service delivery",
        "Coordination challenges delayed operational response",
        "Management requested route modifications",
        "Access restrictions increased service complexity",
        "Property upgrades impacted collection logistics",
        "Operational feedback meeting identified improvement areas",
        "Property leadership requested service enhancement review"
    ]
}

properties = [
    "Sunset Ridge",
    "Maple Court",
    "Green Oaks",
    "Birchwood Commons",
    "Lakeview Terrace",
    "Cedar Park",
    "Willow Creek",
    "Harbor Pointe"
]

recommended_actions = [
    "Coordinate with property manager",
    "Update access credentials",
    "Assign backup collection crew",
    "Schedule preventive maintenance",
    "Review route optimization plan",
    "Increase staffing coverage",
    "Conduct resident awareness campaign",
    "Perform operational audit",
    "Escalate issue to regional manager",
    "Monitor performance for next 7 days"
]

severity_levels = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

property_issue_weights = {
    "Sunset Ridge": {
        "Resident Behavior": 0.6,
        "Gate Access": 0.25,
        "Property Management": 0.15
    },

    "Green Oaks": {
        "Staffing": 0.6,
        "Route Planning": 0.25,
        "Vehicle": 0.15
    },

    "Maple Court": {
        "Vehicle": 0.55,
        "Weather": 0.30,
        "Route Planning": 0.15
    },

    "Birchwood Commons": {
        "Property Management": 0.55,
        "Resident Behavior": 0.30,
        "Gate Access": 0.15
    },

    "Lakeview Terrace": {
        "Weather": 0.55,
        "Route Planning": 0.30,
        "Resident Behavior": 0.15
    },

    "Cedar Park": {
        "Route Planning": 0.55,
        "Staffing": 0.30,
        "Property Management": 0.15
    },

    "Willow Creek": {
        "Gate Access": 0.55,
        "Vehicle": 0.30,
        "Staffing": 0.15
    },

    "Harbor Pointe": {
        "Vehicle": 0.55,
        "Staffing": 0.30,
        "Weather": 0.15
    }
}

# Building property_issue_map from property_issue_weights
property_issue_map = {
    prop: list(issues.keys())
    for prop, issues in property_issue_weights.items()
}

# Generating Notes

note_rows = []

base_date = datetime(2025, 1, 1)

for i in range(5000):

    prop = random.choice(properties)

    issue_type = random.choice(property_issue_map[prop])

    note_text = random.choice(issue_templates[issue_type])

    note_rows.append({
        "note_id": f"N{i+1:05d}",
        "property_name": prop,
        "date": (
            base_date +
            timedelta(days=random.randint(0, 180))
        ).strftime("%Y-%m-%d"),
        "issue_type": issue_type,
        "severity": random.choice(severity_levels),
        "note_text": note_text,
        "recommended_action": random.choice(recommended_actions)
    })     

# Creating DataFrame

pdf_notes = pd.DataFrame(note_rows)

df_notes = spark.createDataFrame(pdf_notes)

print(f"Created {df_notes.count()} rows")

display(df_notes.limit(10))

Created 5000 rows


note_id,property_name,date,issue_type,severity,note_text,recommended_action
N00001,Birchwood Commons,2025-04-01,Resident Behavior,High,Excess waste volume exceeded expected capacity,Perform operational audit
N00002,Sunset Ridge,2025-04-24,Resident Behavior,Medium,Missed communication led to resident dissatisfaction,Conduct resident awareness campaign
N00003,Willow Creek,2025-05-11,Staffing,Low,Insufficient weekend staffing caused backlog,Monitor performance for next 7 days
N00004,Green Oaks,2025-02-26,Staffing,Low,Staff shortage on route caused missed pickups,Schedule preventive maintenance
N00005,Birchwood Commons,2025-02-16,Resident Behavior,Low,Late bin placement caused missed pickups,Conduct resident awareness campaign
N00006,Sunset Ridge,2025-06-28,Property Management,Low,Property upgrades impacted collection logistics,Update access credentials
N00007,Green Oaks,2025-01-28,Vehicle,Low,Fleet availability issues caused route reassignment,Schedule preventive maintenance
N00008,Birchwood Commons,2025-03-10,Resident Behavior,Critical,Improper waste disposal created collection challenges,Review route optimization plan
N00009,Willow Creek,2025-01-04,Vehicle,Critical,Vehicle diagnostics required immediate attention,Perform operational audit
N00010,Lakeview Terrace,2025-05-04,Weather,Medium,Poor visibility affected route safety,Increase staffing coverage


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS valetai_db")
spark.sql("USE valetai_db")
print("Database valetai_db is ready")

Database valetai_db is ready


# Bronze Layer

In [0]:
from pyspark.sql.functions import current_timestamp

df_bronze_logs = df_logs.withColumn("ingested_at", current_timestamp())

df_bronze_logs.write.format("delta").mode("overwrite") \
    .saveAsTable("valetai_db.bronze_waste_logs")

print("Saved bronze_waste_logs:", df_bronze_logs.count(), "rows")

Saved bronze_waste_logs: 8000 rows


In [0]:
df_bronze_notes = df_notes.withColumn("ingested_at", current_timestamp())

df_bronze_notes.write.format("delta").mode("overwrite") \
    .saveAsTable("valetai_db.bronze_operational_notes")

print("Saved bronze_operational_notes:", df_bronze_notes.count(), "rows")

Saved bronze_operational_notes: 5000 rows


# Silver Layer

In [0]:
from pyspark.sql.functions import col, to_date, current_timestamp

df_b_logs = spark.read.table("valetai_db.bronze_waste_logs")

df_silver_logs = (
    df_b_logs
    .dropDuplicates(["collection_id"])              
    .fillna({"bags_collected": 0, "missed_pickups": 0}) 
    .withColumn("collection_date", to_date(col("collection_date"), "yyyy-MM-dd")) # text → real date
    .withColumn("processed_at", current_timestamp())
)

df_silver_logs.write.format("delta").mode("overwrite") \
    .saveAsTable("valetai_db.silver_waste_logs")

print("Saved silver_waste_logs:", df_silver_logs.count(), "rows")
display(df_silver_logs.limit(5))

Saved silver_waste_logs: 8000 rows


collection_id,property_id,property_name,collection_date,route_id,bags_collected,missed_pickups,completion_time,collector_id,region,ingested_at,processed_at
C001002,P002,Maple Court,2025-06-04,R11,87,2,18:02,D17,Midwest,2026-06-25T06:46:29.225Z,2026-06-25T10:09:37.067Z
C001004,P005,Lakeview Terrace,2025-05-31,R24,76,0,17:11,D16,Southeast,2026-06-25T06:46:29.225Z,2026-06-25T10:09:37.067Z
C001010,P002,Maple Court,2025-05-24,R13,119,0,22:36,D02,Midwest,2026-06-25T06:46:29.225Z,2026-06-25T10:09:37.067Z
C001012,P008,Harbor Pointe,2025-02-22,R11,143,0,22:38,D11,West,2026-06-25T06:46:29.225Z,2026-06-25T10:09:37.067Z
C001019,P007,Willow Creek,2025-01-17,R20,69,0,14:17,D18,Southeast,2026-06-25T06:46:29.225Z,2026-06-25T10:09:37.067Z


In [0]:
from pyspark.sql.functions import trim, initcap

df_b_notes= spark.read.table("valetai_db.bronze_operational_notes")

df_silver_notes = (
    df_b_notes
    .dropDuplicates(["note_id"])
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .dropna(subset=["note_text"])              # drop rows with no text - useless for RAG
    .withColumn("processed_at", current_timestamp())
)

df_silver_notes.write.format("delta").mode("overwrite") \
    .saveAsTable("valetai_db.silver_operational_notes")

print("Saved silver_operational_notes:", df_silver_notes.count(), "rows")

Saved silver_operational_notes: 5000 rows


# Gold Layer

In [0]:
from pyspark.sql.functions import sum as _sum, avg, count, to_timestamp, unix_timestamp

df_s_logs = spark.read.table("valetai_db.silver_waste_logs")

df_with_minutes = df_s_logs.withColumn(
    "completion_minutes",
    (col("completion_time").substr(1,2).cast("int") * 60) +
     col("completion_time").substr(4,2).cast("int")
)

df_gold = (
    df_with_minutes
    .groupBy("property_name", "property_id")
    .agg(
        count("collection_id").alias("total_collections"),
        _sum("missed_pickups").alias("total_missed_pickups"),
        _sum("bags_collected").alias("total_bags_collected"),
        avg("completion_minutes").alias("avg_completion_minutes"),
    )
    .withColumn(
        "missed_pickup_rate",
        (col("total_missed_pickups") / col("total_collections"))
    )
)

(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("valetai_db.gold_property_metrics")
)

print("Saved gold_property_metrics:", df_gold.count(), "rows (one per property)")
display(df_gold.orderBy(col("missed_pickup_rate").desc()))

Saved gold_property_metrics: 8 rows (one per property)


property_name,property_id,total_collections,total_missed_pickups,total_bags_collected,avg_completion_minutes,missed_pickup_rate
Green Oaks,P003,1029,559,101949,1116.821185617104,0.543245869776482
Harbor Pointe,P008,1000,531,99260,1106.582,0.531
Sunset Ridge,P001,973,515,97036,1110.546762589928,0.5292908530318602
Cedar Park,P006,986,472,100676,1108.2423935091279,0.4787018255578093
Maple Court,P002,971,462,97709,1116.870236869207,0.47579814624098865
Lakeview Terrace,P005,1051,498,103526,1107.0085632730732,0.4738344433872502
Birchwood Commons,P004,971,458,98418,1104.7548918640578,0.47167868177136973
Willow Creek,P007,1019,466,102611,1107.8253189401373,0.4573110893032385


In [0]:
tables = [
    "bronze_waste_logs", "bronze_operational_notes",
    "silver_waste_logs", "silver_operational_notes",
    "gold_property_metrics",
]

print("=== PIPELINE CHECK ===")
for t in tables:
    df_check = spark.read.table(f"valetai_db.{t}")
    print(f"{t:32s} -> {df_check.count()} rows")
print("All 5 tables exist and are readable.")

=== PIPELINE CHECK ===
bronze_waste_logs                -> 8000 rows
bronze_operational_notes         -> 5000 rows
silver_waste_logs                -> 8000 rows
silver_operational_notes         -> 5000 rows
gold_property_metrics            -> 8 rows
All 5 tables exist and are readable.


# Exporting to csv

In [0]:
# Creating a volume first
spark.sql("CREATE VOLUME IF NOT EXISTS valetai_db.exports")

DataFrame[]

In [0]:
df_gold_export = spark.read.table("valetai_db.gold_property_metrics")
df_notes_export = spark.read.table("valetai_db.silver_operational_notes")
df_logs_export = spark.read.table("valetai_db.silver_waste_logs")

df_gold_export.toPandas().to_csv("/Volumes/workspace/valetai_db/exports/gold_property_metrics.csv", index=False)
df_notes_export.toPandas().to_csv("/Volumes/workspace/valetai_db/exports/silver_operational_notes.csv", index=False)
df_logs_export.toPandas().to_csv("/Volumes/workspace/valetai_db/exports/silver_waste_logs.csv", index=False)
print("Exported to Volume")

Exported to Volume


In [0]:
display(df_gold_export)


property_name,property_id,total_collections,total_missed_pickups,total_bags_collected,avg_completion_minutes,missed_pickup_rate
Birchwood Commons,P004,971,458,98418,1104.7548918640578,0.47167868177136973
Cedar Park,P006,986,472,100676,1108.2423935091279,0.4787018255578093
Green Oaks,P003,1029,559,101949,1116.821185617104,0.543245869776482
Maple Court,P002,971,462,97709,1116.870236869207,0.47579814624098865
Lakeview Terrace,P005,1051,498,103526,1107.0085632730732,0.4738344433872502
Willow Creek,P007,1019,466,102611,1107.8253189401373,0.4573110893032385
Harbor Pointe,P008,1000,531,99260,1106.582,0.531
Sunset Ridge,P001,973,515,97036,1110.546762589928,0.5292908530318602


In [0]:
df_notes_export = spark.read.table("valetai_db.silver_operational_notes")
display(df_notes_export)

note_id,property_name,date,issue_type,severity,note_text,recommended_action,ingested_at,processed_at
N04006,Green Oaks,2025-05-22,Staffing,Critical,Staff reassignment impacted scheduled routes,Assign backup collection crew,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04019,Harbor Pointe,2025-06-14,Vehicle,High,Replacement vehicle arrived later than expected,Update access credentials,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04027,Sunset Ridge,2025-03-02,Gate Access,High,Incorrect gate code provided by property management,Coordinate with property manager,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04050,Willow Creek,2025-02-04,Gate Access,Low,Community gate remained locked during scheduled pickup window,Perform operational audit,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04063,Sunset Ridge,2025-02-10,Property Management,Low,Management requested route modifications,Coordinate with property manager,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04069,Lakeview Terrace,2025-06-03,Weather,High,Extreme temperatures slowed collection activities,Increase staffing coverage,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04071,Harbor Pointe,2025-02-09,Staffing,Critical,Temporary workers required additional supervision,Coordinate with property manager,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04083,Maple Court,2025-06-18,Weather,Medium,Heavy rain delayed collection schedule across region,Perform operational audit,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04096,Harbor Pointe,2025-04-05,Weather,Medium,Lightning risk delayed evening collections,Schedule preventive maintenance,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z
N04099,Birchwood Commons,2025-03-12,Resident Behavior,Critical,Excess waste volume exceeded expected capacity,Review route optimization plan,2026-06-25T10:09:16.280Z,2026-06-25T10:09:40.811Z


In [0]:
display(df_logs_export)

collection_id,property_id,property_name,collection_date,route_id,bags_collected,missed_pickups,completion_time,collector_id,region,ingested_at,processed_at
C001002,P002,Maple Court,2025-06-04,R11,87,2,18:02,D17,Midwest,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001004,P005,Lakeview Terrace,2025-05-31,R24,76,0,17:11,D16,Southeast,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001010,P002,Maple Court,2025-05-24,R13,119,0,22:36,D02,Midwest,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001012,P008,Harbor Pointe,2025-02-22,R11,143,0,22:38,D11,West,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001019,P007,Willow Creek,2025-01-17,R20,69,0,14:17,D18,Southeast,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001026,P006,Cedar Park,2025-06-06,R24,127,0,19:40,D09,West,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001036,P004,Birchwood Commons,2025-03-08,R24,106,2,18:12,D01,Northeast,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001038,P007,Willow Creek,2025-04-21,R14,55,1,22:29,D16,Northeast,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001078,P005,Lakeview Terrace,2025-06-22,R21,144,0,18:12,D20,Midwest,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
C001079,P001,Sunset Ridge,2025-05-20,R22,78,2,18:52,D12,West,2026-06-25T06:46:29.225Z,2026-06-25T10:09:32.948Z
